In [18]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn import linear_model
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time

In [19]:
med_df = pd.read_csv('heart_failure_clinical_records_dataset.csv')

### Task Description
For this assignment I used the Heart Failure Clinical Records dataset, which contains information on 299 patients and whether they died during the follow-up period. The goal is to use the 13 clinical features to predict the binary outcome DEATH_EVENT. I chose the Lasso model because it is simple, easy to interpret, and allows me to explore how regularization strength affects linear patterns in the data. Lasso also makes it clear when features are not strong predictors since it can shrink unimportant coefficients toward zero. This model provides a good baseline for understanding the limits of linear prediction on this dataset.

In [20]:
med_df.head(5)

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


In [21]:
med_df.describe()

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
count,299.000000,299.000000,299.000000,299.000000,299.000000,299.000000,299.000000,299.00000,299.000000,299.000000,299.00000,299.000000,299.00000
mean,60.833893,0.431438,581.839465,0.418060,38.083612,0.351171,263358.029264,1.39388,136.625418,0.648829,0.32107,130.260870,0.32107
std,11.894809,0.496107,970.287881,0.494067,11.834841,0.478136,97804.236869,1.03451,4.412477,0.478136,0.46767,77.614208,0.46767
min,40.000000,0.000000,23.000000,0.000000,14.000000,0.000000,25100.000000,0.50000,113.000000,0.000000,0.00000,4.000000,0.00000
25%,51.000000,0.000000,116.500000,0.000000,30.000000,0.000000,212500.000000,0.90000,134.000000,0.000000,0.00000,73.000000,0.00000
50%,60.000000,0.000000,250.000000,0.000000,38.000000,0.000000,262000.000000,1.10000,137.000000,1.000000,0.00000,115.000000,0.00000
75%,70.000000,1.000000,582.000000,1.000000,45.000000,1.000000,303500.000000,1.40000,140.000000,1.000000,1.00000,203.000000,1.00000
max,95.000000,1.000000,7861.000000,1.000000,80.000000,1.000000,850000.000000,9.40000,148.000000,1.000000,1.00000,285.000000,1.00000


In [22]:
med_X = med_df.drop('DEATH_EVENT', axis=1)
med_y = med_df['DEATH_EVENT']

In [25]:
med_X_train, med_X_test, med_y_train, med_y_test = train_test_split(
    med_X, med_y, test_size=0.2, random_state=74)

In [26]:
default_lasso = Lasso()
default_lasso.fit(med_X_train, med_y_train)
default_lasso.score(med_X_test, med_y_test)

0.25928650540177467

### Default Model Fit
The default Lasso model (alpha = 1.0) scored about 0.259 on the test set. This means the model explains around 25% of the variation in the outcome, which is a modest amount but still indicates some relationship between the features and the target. Because the regularization is strong at the default setting, many feature coefficients are pushed down, which keeps the model simple. Even though the score is not high, it provides a reasonable starting point to compare against tuned models. Overall, the default model captures some useful signal but still underfits the data.

In [29]:
lasso = Lasso()
params_lasso = {
    'alpha': [0.001, 0.01, 0.05, 0.1, 0.25, 0.5, 1.0],
    'max_iter': [1000, 5000],
}

grid = GridSearchCV(lasso, params_lasso)
grid.fit(med_X_train, med_y_train)

,estimator,Lasso()
,param_grid,"{'alpha': [0.001, 0.01, ...], 'max_iter': [1000, 5000]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,alpha,0.05


### Parameters
Since the default model underfit the data, I chose a parameter grid with smaller alpha values such as 0.001, 0.01, and 0.05. These values let the model keep more information instead of shrinking everything too much. I also included two different max_iter values (1000 and 5000) to ensure the model had enough time to converge. This grid is small but covers a range of alpha values from very light to very strong penalties. It gives the model a fair chance to see whether it performs better with a looser or tighter penalty.

In [30]:
grid.score(med_X_test, med_y_test)

0.41663147910690124

In [31]:
regr_grid = pd.DataFrame(grid.cv_results_)
regr_grid.head(5)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_alpha,param_max_iter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.002360,0.001307,0.001391,0.000334,0.001,1000,"{'alpha': 0.001, 'max_iter': 1000}",0.110974,0.253881,0.297562,0.288254,0.384141,0.266962,0.089021,11
1,0.002065,0.000987,0.001224,0.000441,0.001,5000,"{'alpha': 0.001, 'max_iter': 5000}",0.110974,0.253881,0.297562,0.288254,0.384141,0.266962,0.089021,11
2,0.001195,0.000106,0.000763,0.000029,0.010,1000,"{'alpha': 0.01, 'max_iter': 1000}",0.185762,0.316359,0.310808,0.304307,0.420126,0.307473,0.074303,3
3,0.001057,0.000015,0.000708,0.000003,0.010,5000,"{'alpha': 0.01, 'max_iter': 5000}",0.185762,0.316359,0.310808,0.304307,0.420126,0.307473,0.074303,3
4,0.001158,0.000136,0.000789,0.000134,0.050,1000,"{'alpha': 0.05, 'max_iter': 1000}",0.171060,0.368866,0.279972,0.315141,0.434399,0.313888,0.088428,1


In [32]:
grid.best_params_

{'alpha': 0.05, 'max_iter': 1000}

In [33]:
grid.best_score_

np.float64(0.3138876591471159)

In [34]:
best_lasso = grid.best_estimator_

### Best Model vs the Default Model
Comparing the two models shows that tuning the alpha parameter improved performance from 0.259 to 0.417, which is a noticeable increase. This suggests that the default regularization strength held the model back and a smaller penalty allowed it to learn more useful patterns. The improvement also shows that there are some linear relationships in the dataset, but the default model was too restricted to capture them well. The tuned model is still not extremely accurate, but it performs meaningfully better than the default and serves as a stronger baseline.

In [35]:
lasso = linear_model.Lasso(alpha=.25)
lasso.fit(med_X_train, med_y_train)
lasso.score(med_X_test, med_y_test)

0.3568732355204355

In [36]:
lasso = linear_model.Lasso(alpha=.15)
lasso.fit(med_X_train, med_y_train)
lasso.score(med_X_test, med_y_test)

0.375654291829979

In [37]:
lasso = linear_model.Lasso(alpha=.05)
lasso.fit(med_X_train, med_y_train)
lasso.score(med_X_test, med_y_test)

0.41663147910690124

### Cross-Validation Results
When I looked at the cross-validation results from GridSearchCV, the scores for the different alpha values were all fairly close to each other. Some alphas performed slightly better than others, but the differences were small, showing that the model’s performance only changes a little depending on the regularization strength. The grid search also ran quickly because the dataset is small and Lasso is a simple model, so time did not vary much across the different parameter combinations. Overall, the CV results suggest that tuning the parameters helps, but only by a modest amount, and most settings produce similar performance.

In [38]:
lasso = Lasso()
grid_5 = GridSearchCV(lasso, params_lasso, cv=5)
grid_5.fit(med_X_train, med_y_train)

,estimator,Lasso()
,param_grid,"{'alpha': [0.001, 0.01, ...], 'max_iter': [1000, 5000]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,alpha,0.05


In [39]:
best_lasso = grid_5.best_estimator_

In [40]:
best_lasso.score(med_X_test, med_y_test)

0.41663147910690124

In [41]:
regr_5_grid = pd.DataFrame(grid_5.cv_results_)
regr_5_grid.head(5)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_alpha,param_max_iter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.003221,0.001948,0.002129,0.001455,0.001,1000,"{'alpha': 0.001, 'max_iter': 1000}",0.110974,0.253881,0.297562,0.288254,0.384141,0.266962,0.089021,11
1,0.001228,0.000114,0.000806,0.000061,0.001,5000,"{'alpha': 0.001, 'max_iter': 5000}",0.110974,0.253881,0.297562,0.288254,0.384141,0.266962,0.089021,11
2,0.001074,0.000032,0.000735,0.000028,0.010,1000,"{'alpha': 0.01, 'max_iter': 1000}",0.185762,0.316359,0.310808,0.304307,0.420126,0.307473,0.074303,3
3,0.001050,0.000008,0.000740,0.000061,0.010,5000,"{'alpha': 0.01, 'max_iter': 5000}",0.185762,0.316359,0.310808,0.304307,0.420126,0.307473,0.074303,3
4,0.001079,0.000042,0.000722,0.000013,0.050,1000,"{'alpha': 0.05, 'max_iter': 1000}",0.171060,0.368866,0.279972,0.315141,0.434399,0.313888,0.088428,1


In [42]:
lasso = Lasso()
grid_10 = GridSearchCV(lasso, params_lasso, cv=10)
grid_10.fit(med_X_train, med_y_train)

,estimator,Lasso()
,param_grid,"{'alpha': [0.001, 0.01, ...], 'max_iter': [1000, 5000]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,10
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,alpha,0.05


In [43]:
best_lasso = grid_10.best_estimator_

In [44]:
best_lasso.score(med_X_test, med_y_test)

0.41663147910690124

In [45]:
regr_10_grid = pd.DataFrame(grid_10.cv_results_)
regr_10_grid.head(5)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_alpha,param_max_iter,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,split5_test_score,split6_test_score,split7_test_score,split8_test_score,split9_test_score,mean_test_score,std_test_score,rank_test_score
0,0.002208,0.001642,0.001360,0.001573,0.001,1000,"{'alpha': 0.001, 'max_iter': 1000}",-0.145741,0.236087,0.096411,0.506055,0.216099,0.372471,0.272623,0.363003,0.255472,0.452193,0.262467,0.177411,11
1,0.001223,0.000097,0.000788,0.000042,0.001,5000,"{'alpha': 0.001, 'max_iter': 5000}",-0.145741,0.236087,0.096411,0.506055,0.216099,0.372471,0.272623,0.363003,0.255472,0.452193,0.262467,0.177411,11
2,0.001077,0.000031,0.000726,0.000032,0.010,1000,"{'alpha': 0.01, 'max_iter': 1000}",0.025957,0.254682,0.148483,0.557980,0.225818,0.362326,0.267953,0.377516,0.276968,0.460579,0.295826,0.144648,3
3,0.001058,0.000034,0.000724,0.000044,0.010,5000,"{'alpha': 0.01, 'max_iter': 5000}",0.025957,0.254682,0.148483,0.557980,0.225818,0.362326,0.267953,0.377516,0.276968,0.460579,0.295826,0.144648,3
4,0.001025,0.000005,0.000698,0.000003,0.050,1000,"{'alpha': 0.05, 'max_iter': 1000}",0.051318,0.262327,0.190111,0.567749,0.197007,0.351114,0.257609,0.404053,0.280593,0.464150,0.302603,0.141466,1


### Cross Validation Parameters
When I changed the number of folds from 5 to 10, the results stayed almost the same. The best alpha chosen by 10-fold CV was the same as the one chosen by 5-fold CV, and the test score was also very similar. The 10-fold grid search took a bit longer to run because it trains more models, but the increase in accuracy was not meaningful. This shows that changing the cross-validation method does not change the overall conclusion: the model improves slightly when tuned, but the differences are small, and the dataset does not have strong linear structure for Lasso to take advantage of.